# 02 — ResNet50 feature extraction (TensorFlow/Keras)

Extracts 2048-dimensional ResNet50 features (ImageNet-1k weights, global average pooling) for all
80,042 records of the metadata table. These features are used for the ResNet50 configuration in the paper.

Pipeline (Section IV-A of the paper):
- each image is resized to 224 × 224 **without preserving the aspect ratio**, with Keras' default
  nearest-neighbour interpolation (`tf.keras.utils.load_img`);
- `keras.applications.resnet50.preprocess_input` (RGB → BGR, ImageNet mean subtraction);
- features are stored in float32 **without L2 normalisation**.

**Environment:** TensorFlow (see `requirements-tf.txt`).

**Input:** `data/metadata/artemis_metadata.parquet` (notebook 01) and the images in `data/wikiart/`.

**Output:** `cache/embeddings/embeddings.npy`, `embedding_index.parquet`, `embedding_info.json`.

In [ ]:
import json
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

warnings.filterwarnings("ignore")
print("TensorFlow:", tf.__version__, "| NumPy:", np.__version__, "| pandas:", pd.__version__)

In [ ]:
import os
from pathlib import Path

# Project root: the repository folder (notebooks/ is one level below it).
# Override with the environment variable ARTEMIS_ROOT if the data live elsewhere.
PROJECT_ROOT = Path(os.environ.get("ARTEMIS_ROOT", Path.cwd().parent)).resolve()

MODEL_NAME    = "ResNet50"
DATASET_NAME  = "WikiArt"
EMBEDDING_DIM = 2048

EXTRACTOR_SPEC = {
    "framework": "tensorflow-keras",
    "model": "ResNet50",
    "weights": "imagenet",
    "pooling": "avg",
    "input_size": 224,
    "preprocess": "keras.applications.resnet50.preprocess_input",
    "resize": "tf.keras.utils.load_img(target_size=(224, 224), interpolation='nearest')",
    "embedding_dimension": 2048,
}

DATA_DIR      = PROJECT_ROOT / "data"
WIKIART_DIR   = DATA_DIR / "wikiart"
METADATA_DIR  = DATA_DIR / "metadata"
EMBEDDING_DIR = PROJECT_ROOT / "cache" / "embeddings"
EXCEL_DIR     = PROJECT_ROOT / "results" / "excel"
for d in (EMBEDDING_DIR, EXCEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

METADATA_FILE        = METADATA_DIR / "artemis_metadata.parquet"
EMBEDDINGS_FILE      = EMBEDDING_DIR / "embeddings.npy"
EMBEDDING_INDEX_FILE = EMBEDDING_DIR / "embedding_index.parquet"
EMBEDDING_INFO_FILE  = EMBEDDING_DIR / "embedding_info.json"

## System check

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print("Built with CUDA :", tf.test.is_built_with_cuda())
print("GPUs            :", [g.name for g in gpus] or "none (runs on CPU)")

## Load the metadata

In [ ]:
if not METADATA_FILE.exists():
    raise FileNotFoundError(f"{METADATA_FILE} not found. Run notebook 01 first.")
artemis_df = pd.read_parquet(METADATA_FILE)
print(f"Records: {len(artemis_df):,}   Movements: {artemis_df['movement'].nunique()}   "
      f"Artists: {artemis_df['artist'].nunique():,}")

## Model and preprocessing

In [ ]:
resnet_model = ResNet50(weights="imagenet", include_top=False, pooling="avg", input_shape=(224, 224, 3))
resnet_model.trainable = False
print("Output shape:", resnet_model.output_shape, "| parameters:", f"{resnet_model.count_params():,}")


def preprocess_image(image_path):
    """Load an image, resize it to 224x224 (nearest neighbour, aspect ratio not preserved)
    and apply the ResNet50 preprocessing. Returns an array of shape (1, 224, 224, 3)."""
    image = tf.keras.utils.load_img(image_path, target_size=(224, 224), interpolation="nearest")
    image = tf.keras.utils.img_to_array(image)
    image = np.expand_dims(image, axis=0)
    return preprocess_input(image)


# quick test on the first available image
sample_image = next(WIKIART_DIR / f for f in artemis_df["filename"] if (WIKIART_DIR / f).exists())
sample_embedding = resnet_model.predict(preprocess_image(sample_image), verbose=0).squeeze()
print("Test embedding:", sample_embedding.shape, sample_embedding.dtype)

## Extract features for all images

In [ ]:
BATCH_SIZE = 32
image_paths = [WIKIART_DIR / f for f in artemis_df["filename"]]
total_images = len(image_paths)
print(f"Images: {total_images:,}   missing on disk: {sum(not p.exists() for p in image_paths)}")

start_time = time.time()
embeddings, failed_images = [], []
for batch_start in tqdm(range(0, total_images, BATCH_SIZE), desc="Embedding"):
    batch_images = []
    for path in image_paths[batch_start:batch_start + BATCH_SIZE]:
        try:
            batch_images.append(preprocess_image(path)[0])
        except Exception as e:
            failed_images.append({"path": str(path), "error": str(e)})
    if batch_images:
        embeddings.extend(resnet_model.predict(np.stack(batch_images), verbose=0))

embeddings = np.asarray(embeddings, dtype=np.float32)
print(f"Embeddings: {embeddings.shape}   failed: {len(failed_images)}   "
      f"time: {(time.time() - start_time) / 60:.1f} min")

## Index, checks and saving

In [ ]:
embedding_index = artemis_df.copy()
embedding_index.insert(1, "embedding_index", np.arange(len(embedding_index)))
embedding_index = embedding_index[["image_id", "embedding_index", "filename", "artist",
                                   "movement", "title", "subset", "dataset"]]

assert len(embeddings) == len(artemis_df), "Number of embeddings differs from the metadata"
assert embeddings.shape[1] == EMBEDDING_DIM
assert np.isfinite(embeddings).all(), "NaN or Inf in the embeddings"

np.save(EMBEDDINGS_FILE, embeddings.astype(np.float32, copy=False))
embedding_index.to_parquet(EMBEDDING_INDEX_FILE, index=False)

saved = np.load(EMBEDDINGS_FILE, mmap_mode="r")
assert saved.shape == embeddings.shape and len(pd.read_parquet(EMBEDDING_INDEX_FILE)) == len(embedding_index)

embedding_info = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "image_count": int(embeddings.shape[0]),
    "embedding_dimension": int(embeddings.shape[1]),
    "dtype": str(embeddings.dtype),
    "l2_normalized": False,
    "extractor": EXTRACTOR_SPEC,
    "tensorflow_version": tf.__version__,
    "failed_image_count": len(failed_images),
    "embedding_file": EMBEDDINGS_FILE.name,
    "index_file": EMBEDDING_INDEX_FILE.name,
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}
with open(EMBEDDING_INFO_FILE, "w", encoding="utf-8") as f:
    json.dump(embedding_info, f, indent=4, ensure_ascii=False)

print("✓", EMBEDDINGS_FILE.name, embeddings.shape)
print("✓", EMBEDDING_INDEX_FILE.name)
print("✓", EMBEDDING_INFO_FILE.name)

## Optional: verify stored features

Re-extracts a random sample of images and compares them with the stored rows. A minimum cosine
similarity above 0.999 confirms that `embeddings.npy` was produced with this pipeline.

In [ ]:
VERIFY_N = 100
_idx = pd.read_parquet(EMBEDDING_INDEX_FILE)
_E = np.load(EMBEDDINGS_FILE, mmap_mode="r")
_pick = np.random.default_rng(0).choice(len(_idx), size=min(VERIFY_N, len(_idx)), replace=False)

cos, mad = [], []
for i in _pick:
    row = _idx.iloc[i]
    new = resnet_model.predict(preprocess_image(WIKIART_DIR / row["filename"]), verbose=0)[0]
    old = np.asarray(_E[int(row["embedding_index"])])
    cos.append(float(new @ old / (np.linalg.norm(new) * np.linalg.norm(old) + 1e-12)))
    mad.append(float(np.max(np.abs(new - old))))
cos, mad = np.array(cos), np.array(mad)
print(f"Cosine min / mean: {cos.min():.6f} / {cos.mean():.6f}   max |difference|: {mad.max():.5f}")
print("✓ same extractor" if cos.min() > 0.999 else "⚠ deviation: re-create embeddings.npy with this notebook")